# Урок 14. Нейросети и классификация цифр
**Библиотеки:** tensorflow/keras (pip install tensorflow), matplotlib. Если TF не ставится — внизу есть запасной вариант на sklearn.

**Цель:** понять нейрон/слой/активацию и обучить сеть распознавать рукописные цифры.

## Простыми словами
**Нейрон** = наша старая знакомая: w·x + b → активация. То есть один нейрон ≈ маленькая логистическая регрессия.
**Слой (Dense)** = много нейронов параллельно, каждый смотрит на все входы.
**Сеть** = слои друг за другом: выходы одного — входы следующего. Каждый следующий слой
собирает из простых признаков более сложные (пиксели → штрихи → петли → «это восьмёрка»).

**Активация** — нелинейность между слоями. Без неё стопка слоёв схлопывается в одну линейную модель.
- ReLU: max(0, x) — стандарт для скрытых слоёв.
- Softmax: превращает выходы в вероятности классов (сумма = 1) — для последнего слоя.

**Forward propagation** — прогон данных через слои до ответа. **Epoch** — один полный проход по train.
Обучение — тот же gradient descent, только параметров тысячи.

In [ ]:
# Грузим MNIST: 70000 картинок цифр 28x28 пикселей
from tensorflow import keras
import numpy as np
import matplotlib.pyplot as plt

(X_tr, y_tr), (X_te, y_te) = keras.datasets.mnist.load_data()
print("train:", X_tr.shape, "| test:", X_te.shape)   # (60000, 28, 28)

# Смотрим глазами — всегда!
fig, axes = plt.subplots(1, 8, figsize=(12, 2))
for ax, img, lbl in zip(axes, X_tr, y_tr):
    ax.imshow(img, cmap='gray'); ax.set_title(lbl); ax.axis('off')
plt.show()

X_tr = X_tr / 255.0   # пиксели 0..255 -> 0..1 (это и есть scaling для картинок!)
X_te = X_te / 255.0

In [ ]:
model = keras.Sequential([
    keras.layers.Flatten(input_shape=(28, 28)),     # картинка 28x28 -> вектор из 784 чисел
    keras.layers.Dense(128, activation='relu'),      # скрытый слой: 128 нейронов
    keras.layers.Dense(10, activation='softmax'),    # 10 выходов = вероятности цифр 0..9
])
model.compile(optimizer='adam',                              # adam = умный gradient descent
              loss='sparse_categorical_crossentropy',        # cost для многоклассовой задачи
              metrics=['accuracy'])
history = model.fit(X_tr, y_tr, epochs=5, validation_split=0.1, verbose=1)

print("\nTest accuracy:", model.evaluate(X_te, y_te, verbose=0)[1].round(4))

In [ ]:
# Кривые обучения: главный диагностический график нейросетей
plt.plot(history.history['accuracy'], label='train')
plt.plot(history.history['val_accuracy'], label='validation')
plt.xlabel('epoch'); plt.ylabel('accuracy'); plt.legend(); plt.grid(True); plt.show()

# Смотрим на ошибки модели (error analysis)
probs = model.predict(X_te, verbose=0)
preds = probs.argmax(axis=1)            # argmax: класс с максимальной вероятностью
wrong = np.where(preds != y_te)[0][:8]
fig, axes = plt.subplots(1, 8, figsize=(12, 2))
for ax, i in zip(axes, wrong):
    ax.imshow(X_te[i], cmap='gray'); ax.axis('off')
    ax.set_title(f'было {y_te[i]}, сказала {preds[i]}', fontsize=8)
plt.show()

In [ ]:
# ЗАПАСНОЙ ВАРИАНТ без TensorFlow (если не установился): нейросеть из sklearn
# from sklearn.neural_network import MLPClassifier
# from sklearn.datasets import load_digits
# from sklearn.model_selection import train_test_split
# d = load_digits()
# Xtr, Xte, ytr, yte = train_test_split(d.data/16, d.target, test_size=0.2, random_state=42)
# mlp = MLPClassifier(hidden_layer_sizes=(64,), max_iter=300).fit(Xtr, ytr)
# print('test acc:', mlp.score(Xte, yte))

## Что произошло
784 пикселя → 128 нейронов ReLU → 10 вероятностей softmax. За 5 эпох сеть достигла ~97–98% на test.
Кривые train/val идут рядом — переобучения почти нет. На картинках ошибок видно: сеть путается
на действительно кривых цифрах — её ошибки «человечны». Это твой первый шаг в Computer Vision!

## Типичные ошибки
1. Забыть нормировку /255 — сеть учится плохо или не учится.
2. Softmax в скрытых слоях или relu в выходном — путать места активаций.
3. Гнать 100 эпох и не смотреть val-кривую: train растёт, val падает = overfit, пора остановиться.

## Конспект 📓
Нейрон = w·x+b + активация. Dense — слой нейронов. ReLU внутри, softmax на выходе классификации.
compile(optimizer, loss, metrics) → fit(epochs, validation_split) → evaluate.
Картинки нормируем /255. Кривые train/val — главный диагноз.

---
## Мини-задание 💪
Добавь второй скрытый слой Dense(64, relu) и обучи заново. Выросла ли test accuracy? А время обучения?

## 5 проверочных вопросов ❓
1. Из чего состоит один нейрон?
2. Зачем нужна активация между слоями?
3. Почему на выходе softmax и что означают его 10 чисел?
4. Что такое epoch?
5. Как по кривым train/val заметить переобучение?

## Домашнее задание 🏠
Задачи 48–49 из practice_tasks.md. Эксперимент: обучи сеть БЕЗ нормировки /255 и сравни кривые.
